# Bmad Interface Example

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pytao import Tao
import os


from bpy_lattice.elements import save_elements_to_json, load_elements_from_json

from bpy_lattice.tracks import save_tracks_to_json

from bpy_lattice import Lattice

from bpy_lattice.interfaces.bmad import (
    EleKey,
    bpy_elements_from_tao,
    floor_orbit_track_from_tao,
)

In [ ]:
LATTICE_FILE = "lat.bmad"

LATTICE_STR = """no_digested
beginning[beta_a] = 1
beginning[beta_b] = 1
beginning[e_tot] = 1e9
parameter[geometry] = open

marker1: marker
drift1: drift, L = 1
ecollimator1: ecollimator, L = 1, descrip="3DMODEL=cone.blend", x_pitch = .2
instrument1: instrument, L =1 
lcavity1: lcavity, L =1
pipe1: pipe, L = 1
quadrupole1: quadrupole, L = 1
rcollimator1: rcollimator, L = 1
sbend1: sbend, L = 1, angle = 13 * pi/180 !, ref_tilt = pi/2, roll = pi/4
sextupole1: sextupole, L = 1
solenoid1: solenoid, L = 1
thick_multipole1: thick_multipole, L = 1
wiggler1: wiggler, L = 1, l_period = 0.1, n_period = 10
*[x_limit] = .04
*[y_limit] = .02

lat: line = (marker1,
drift1,
ecollimator1,
instrument1,
lcavity1,
pipe1, 
quadrupole1,
rcollimator1,
sbend1,
thick_multipole1,
wiggler1)
use, lat

"""

# Test all keys
# LATTICE_FILE = (
#    "$ACC_ROOT_DIR/regression_tests/tracking_method_test/tracking_method_test.bmad"
# )

In [ ]:
# tao = Tao(lattice_file=LATTICE_FILE, plot='mpl')
tao = Tao.from_lattice_contents(LATTICE_STR, plot="mpl")
tao.plot("floor")

# Lattice

In [ ]:
lattice = Lattice.from_tao(tao)
len(lattice.elements), len(lattice.tracks)

In [ ]:
# Serialize to JSON
lattice.to_json("lat.json")

In [ ]:
# Load from JSON
lattice2 = lattice.from_json("lat.json")

In [ ]:
# Check elements
for ele1, ele2 in zip(lattice.elements, lattice2.elements):
    assert ele1 == ele2

In [ ]:
# Add elements to Blender
library_cache, objects = lattice.add_elements_to_blender(
    catalogue=os.path.expandvars("$BLENDER_CATALOGUE")
)

In [ ]:
?lattice.add_elements_to_blender

In [ ]:
# Add tracks to Blender
lattice.add_tracks_to_blender()

# Elements

In [ ]:
EleKey("ecollimator")

In [ ]:
eles = bpy_elements_from_tao(tao)
for ele in eles:
    ele.cad_model = ""

save_elements_to_json(eles, "elements.json")
ele = eles[0]
ele

In [ ]:
len(eles)

In [ ]:
# Check that they were loaded correctly
eles2 = load_elements_from_json("elements.json")

for ele1, ele2 in zip(eles, eles2):
    assert ele1 == ele2

# Tracks 

In [ ]:
track = floor_orbit_track_from_tao(tao)
track

In [ ]:
save_tracks_to_json([track], "tracks.json")

# Script

```python

from bpy_lattice import Lattice, remap_zx

JSON_FILE ="lat.json"

lattice = Lattice.from_json(JSON_FILE)

# Add elements
lattice.add_elements_to_blender()

# Add tracks (orbit)
lattice.add_tracks_to_blender()

# Remap Z->X
remap_zx()

```

# Cleanup

In [ ]:
for file in [
    "tracks.json",
    "lat.bmad",
    # "lat.json",
    "elements.json",
]:
    os.remove(file)